# Car Market Trends Analysis

Random Forest classification for high-price vs low-price cars.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================

# Load the dataset
df = pd.read_csv("1776311302-P3-Car Market Trends Analysis with Car Dekho Data.csv")

# Create a binary classification target: High Price (1) vs Low Price (0)
# (Classifying cars selling above the median price as 'Premium/High Value')
median_price = df["selling_price"].median()
df["high_price_category"] = (df["selling_price"] > median_price).astype(int)

# ==========================================
# 2. DATA PREPROCESSING
# ==========================================

# Feature selection
num_features = ["year", "km_driven"]
cat_features = ["fuel", "seller_type", "transmission", "owner"]

# Handling Categorical Encoding
df_encoded = pd.get_dummies(df[cat_features], drop_first=True)

# Feature Standardization (Z-score normalization)
scaler = StandardScaler()
scaled_num = pd.DataFrame(
    scaler.fit_transform(df[num_features]), columns=num_features
)

# Combine Processed Features
X = pd.concat([scaled_num, df_encoded], axis=1)
y = df["high_price_category"]

# Data Splitting (80% Train, 20% Test with Stratified Sampling)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ==========================================
# 3. MODEL ARCHITECTURE & TRAINING
# ==========================================

rf_model = RandomForestClassifier(
    n_estimators=100, random_state=42, class_weight="balanced"
)
rf_model.fit(X_train, y_train)

# ==========================================
# 4. EVALUATION & CROSS-VALIDATION
# ==========================================

y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

# Metric Evaluation
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy:  {acc:.2f}")
print(f"Precision: {prec:.2f}")
print(f"Recall:    {rec:.2f}")
print(f"F1-Score:  {f1:.2f}")
print(f"ROC-AUC:   {auc:.2f}")

# 5-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    rf_model, X_train, y_train, cv=cv, scoring="roc_auc"
)
print(f"Mean CV ROC-AUC: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
